<a href="https://colab.research.google.com/github/AhamedJazira-M/Adaptive-RAG/blob/main/RAG_1_Adaptive_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##ADAPTIVE RAG

In [3]:
!pip -q install langchain langchain-groq langchain-community langchain-chroma chromadb pypdf sentence-transformers fastapi uvicorn pyngrok nest_asyncio


In [4]:
!pip -q install -U langchain-huggingface

from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


LOADING, CHUNKING, EMBEDDING, VECTOR DB

In [6]:
import os
import getpass
import json
import re
import time
import threading
import nest_asyncio

from google.colab import files

from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

nest_asyncio.apply()

os.environ["GROQ_API_KEY"] = os.getenv("Groq_AJ") or "Enter_groq_api_key"
uploaded = files.upload()
file_path = next(iter(uploaded))

print("Uploaded:", file_path)

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

loader = PyPDFLoader(file_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["page"] = chunk.metadata.get("page", 0)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="adaptive_rag_v2"
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("Pages:", len(documents))
print("Chunks:", len(chunks))
print("Vector database ready.")

Saving Arun-ICCISS -org- Paper.pdf to Arun-ICCISS -org- Paper.pdf
Uploaded: Arun-ICCISS -org- Paper.pdf


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Pages: 5
Chunks: 21
Vector database ready.


#ADAPTIVE RAG

In [10]:
router_prompt = ChatPromptTemplate.from_template("""
You are the query classifier for an Adaptive RAG system.

Classify the user's question using these fields.

query_type:
- FACTUAL: asks for a specific fact, value, definition, method, result, or detail
- ANALYTICAL: asks why, how, comparison, relationship, cause, interpretation, or synthesis
- OPINION: asks for a recommendation, preference, judgment, or evaluation
- DOCUMENT_CONTEXTUAL: explicitly asks about the uploaded document, paper, PDF, abstract,
  introduction, methodology, results, discussion, conclusion, or other document section

scope:
- LOCAL: answer can be supported by a small number of relevant chunks
- GLOBAL: requires information from multiple parts or pages of the document

retrieval_strategy:
- PRECISE: normal semantic retrieval
- BROAD: retrieve more chunks for analysis
- PAGE_AWARE: prioritize a specific document page/section
- GLOBAL_COVERAGE: retrieve information across the document
- DIRECT: document is not needed

target_page:
- Use 1-based page number when a specific page/section is identifiable.
- For "abstract", normally use page 1.
- For "introduction", normally prioritize early pages.
- For "conclusion", normally prioritize the final pages.
- Use null if there is no specific page.

needs_document:
- true if the uploaded document is needed
- false if the question can be answered without it

Return ONLY valid JSON.

Example:
{{
  "query_type": "DOCUMENT_CONTEXTUAL",
  "scope": "LOCAL",
  "retrieval_strategy": "PAGE_AWARE",
  "target_page": 1,
  "needs_document": true
}}

Question:
{question}
""")

router_chain = router_prompt | llm | StrOutputParser()


grader_prompt = ChatPromptTemplate.from_template("""
You are a relevance grader for a RAG system.

Determine whether the supplied context contains enough relevant information
to answer the question.

Return ONLY:
RELEVANT
or
INSUFFICIENT

Question:
{question}

Context:
{context}
""")

grader_chain = grader_prompt | llm | StrOutputParser()


answer_prompt = ChatPromptTemplate.from_template("""
You are answering a question using retrieved document evidence.

Query type: {query_type}

Rules:

1. FACTUAL
   Give the factual answer supported by the context.

2. ANALYTICAL
   Explain the reasoning using evidence from the context.
   Distinguish documented facts from your interpretation.

3. OPINION
   Do not pretend that an opinion is a fact.
   Give an evidence-based assessment and clearly identify the basis
   for the assessment.

4. DOCUMENT_CONTEXTUAL
   Answer specifically from the uploaded document.
   If the question asks about a section such as the abstract,
   focus on that section.

5. If the context is insufficient, say:
   "The retrieved document context does not contain enough information
   to answer this reliably."

Do not invent information.

Context:
{context}

Question:
{question}

Answer:
""")

answer_chain = answer_prompt | llm | StrOutputParser()


direct_prompt = ChatPromptTemplate.from_template("""
Answer the question directly and concisely.

Question:
{question}

Answer:
""")

direct_chain = direct_prompt | llm | StrOutputParser()


def parse_router_output(text):
    text = text.strip()

    text = re.sub(r"```json", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text)

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        return {
            "query_type": "DOCUMENT_CONTEXTUAL",
            "scope": "LOCAL",
            "retrieval_strategy": "PRECISE",
            "target_page": None,
            "needs_document": True
        }

    try:
        result = json.loads(match.group(0))

        return {
            "query_type": result.get(
                "query_type",
                "DOCUMENT_CONTEXTUAL"
            ),
            "scope": result.get(
                "scope",
                "LOCAL"
            ),
            "retrieval_strategy": result.get(
                "retrieval_strategy",
                "PRECISE"
            ),
            "target_page": result.get(
                "target_page",
                None
            ),
            "needs_document": result.get(
                "needs_document",
                True
            )
        }

    except Exception:
        return {
            "query_type": "DOCUMENT_CONTEXTUAL",
            "scope": "LOCAL",
            "retrieval_strategy": "PRECISE",
            "target_page": None,
            "needs_document": True
        }


def page_number(doc):
    return int(doc.metadata.get("page", 0)) + 1


def get_page_chunks(target_page):
    return [
        doc for doc in chunks
        if page_number(doc) == target_page
    ]


def global_retrieval(question):
    """
    Retrieve across the whole document while maintaining page coverage.
    """

    total_pages = len(documents)

    selected = []

    # Retrieve a broad set of semantically relevant chunks.
    broad_docs = vectorstore.similarity_search(
        question,
        k=min(len(chunks), max(12, total_pages * 3))
    )

    # First select relevant chunks while maintaining page diversity.
    page_counts = {}

    for doc in broad_docs:
        p = page_number(doc)

        if page_counts.get(p, 0) < 2:
            selected.append(doc)
            page_counts[p] = page_counts.get(p, 0) + 1

    # If some pages were not represented, add their first chunk.
    represented_pages = set(page_counts.keys())

    for p in range(1, total_pages + 1):
        if p not in represented_pages:
            page_docs = get_page_chunks(p)

            if page_docs:
                selected.append(page_docs[0])

    return selected


def page_aware_retrieval(question, target_page):
    """
    Retrieve from a specific page when the query refers to
    a document section such as abstract/introduction/conclusion.
    """

    if target_page is None:
        return vectorstore.similarity_search(question, k=5)

    page_docs = get_page_chunks(target_page)

    if not page_docs:
        return vectorstore.similarity_search(question, k=5)

    # If the target page is small, use all its chunks.
    if len(page_docs) <= 5:
        return page_docs

    # Otherwise rank chunks from that page.
    page_store = Chroma.from_documents(
        documents=page_docs,
        embedding=embeddings,
        collection_name=f"temporary_page_{target_page}"
    )

    return page_store.similarity_search(
        question,
        k=min(5, len(page_docs))
    )


def adaptive_rag(question):

    # Step 1: classify the query
    routing_text = router_chain.invoke({
        "question": question
    })

    route = parse_router_output(routing_text)

    query_type = route["query_type"]
    scope = route["scope"]
    strategy = route["retrieval_strategy"]
    target_page = route["target_page"]
    needs_document = route["needs_document"]

    # Step 2: direct answer if document is not needed
    if not needs_document or strategy == "DIRECT":
        answer = direct_chain.invoke({
            "question": question
        })

        return {
            "mode": "direct",
            "query_type": query_type,
            "scope": scope,
            "retrieval_strategy": "DIRECT",
            "answer": answer,
            "sources": []
        }

    # Step 3: choose adaptive retrieval mechanism
    if strategy == "PAGE_AWARE":
        docs = page_aware_retrieval(
            question,
            target_page
        )

    elif strategy == "GLOBAL_COVERAGE" or scope == "GLOBAL":
        docs = global_retrieval(question)

    elif strategy == "BROAD":
        docs = vectorstore.similarity_search(
            question,
            k=min(10, len(chunks))
        )

    else:
        docs = vectorstore.similarity_search(
            question,
            k=5
        )

    # Step 4: build context
    context_parts = []

    for doc in docs:
        context_parts.append(
            f"[Page {page_number(doc)}]\n"
            f"{doc.page_content}"
        )

    context = "\n\n".join(context_parts)

    # Step 5: relevance grading
    grade = grader_chain.invoke({
        "question": question,
        "context": context
    }).strip().upper()

    # Step 6: fallback if first retrieval is insufficient
    if "INSUFFICIENT" in grade:

        fallback_docs = vectorstore.similarity_search(
            question,
            k=min(10, len(chunks))
        )

        fallback_context = "\n\n".join(
            f"[Page {page_number(doc)}]\n{doc.page_content}"
            for doc in fallback_docs
        )

        fallback_grade = grader_chain.invoke({
            "question": question,
            "context": fallback_context
        }).strip().upper()

        if "RELEVANT" in fallback_grade:
            docs = fallback_docs
            context = fallback_context
            grade = fallback_grade

    # Step 7: generate final answer
    answer = answer_chain.invoke({
        "context": context,
        "question": question,
        "query_type": query_type
    })

    return {
        "mode": "retrieval",
        "query_type": query_type,
        "scope": scope,
        "retrieval_strategy": strategy,
        "target_page": target_page,
        "relevance": grade,
        "answer": answer,
        "sources": sorted(
            list(set(page_number(doc) for doc in docs))
        )
    }




test_questions = [
    "What does the abstract say?"
]

for question in test_questions:
    print("\n" + "-" * 60)
    print("Question:", question)

    result = adaptive_rag(question)



------------------------------------------------------------
Question: What does the abstract say?


#FAST API

In [12]:
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from google.colab import output
import uvicorn
import threading
import socket
import time
import asyncio


app = FastAPI(title="Adaptive RAG")


class QuestionRequest(BaseModel):
    question: str


@app.get("/", response_class=HTMLResponse)
def home():
    return """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Adaptive RAG</title>

        <style>
            body {
                font-family: Arial, sans-serif;
                max-width: 950px;
                margin: 40px auto;
                padding: 20px;
                background: #f7f7f7;
            }

            h1 {
                margin-bottom: 5px;
            }

            .subtitle {
                color: #666;
                margin-bottom: 25px;
            }

            textarea {
                width: 100%;
                height: 130px;
                padding: 14px;
                font-size: 16px;
                border: 1px solid #ccc;
                border-radius: 8px;
                box-sizing: border-box;
                resize: vertical;
            }

            button {
                margin-top: 12px;
                padding: 12px 28px;
                font-size: 16px;
                border: none;
                border-radius: 7px;
                cursor: pointer;
            }

            .info {
                display: grid;
                grid-template-columns: repeat(2, 1fr);
                gap: 12px;
                margin-top: 25px;
            }

            .card {
                padding: 15px;
                background: white;
                border: 1px solid #ddd;
                border-radius: 8px;
            }

            .label {
                color: #777;
                font-size: 13px;
                margin-bottom: 5px;
            }

            .value {
                font-weight: bold;
                font-size: 15px;
            }

            .answer-box {
                margin-top: 25px;
                padding: 20px;
                background: white;
                border: 1px solid #ddd;
                border-radius: 8px;
            }

            .answer-title {
                font-size: 18px;
                font-weight: bold;
                margin-bottom: 12px;
            }

            #answer {
                white-space: pre-wrap;
                line-height: 1.7;
            }

            .loading {
                opacity: 0.6;
            }
        </style>
    </head>

    <body>

        <h1>Adaptive RAG</h1>

        <div class="subtitle">
            Ask questions about your uploaded PDF
        </div>

        <textarea
            id="question"
            placeholder="Example: What does the abstract say?"
        ></textarea>

        <br>

        <button onclick="askQuestion()">
            Ask Question
        </button>

        <div class="info">

            <div class="card">
                <div class="label">Query Type</div>
                <div class="value" id="queryType">-</div>
            </div>

            <div class="card">
                <div class="label">Scope</div>
                <div class="value" id="scope">-</div>
            </div>

            <div class="card">
                <div class="label">Retrieval Strategy</div>
                <div class="value" id="retrieval">-</div>
            </div>

            <div class="card">
                <div class="label">Target Page</div>
                <div class="value" id="page">-</div>
            </div>

            <div class="card">
                <div class="label">Source Pages</div>
                <div class="value" id="sources">-</div>
            </div>

            <div class="card">
                <div class="label">Relevance</div>
                <div class="value" id="relevance">-</div>
            </div>

        </div>

        <div class="answer-box">

            <div class="answer-title">
                Answer
            </div>

            <div id="answer">
                Ask a question to get an answer.
            </div>

        </div>


        <script>

        async function askQuestion() {

            const question =
                document.getElementById("question").value.trim();

            const answer =
                document.getElementById("answer");

            if (!question) {

                alert("Please enter a question.");

                return;
            }


            answer.classList.add("loading");

            answer.innerText = "Thinking...";


            document.getElementById("queryType").innerText = "-";
            document.getElementById("scope").innerText = "-";
            document.getElementById("retrieval").innerText = "-";
            document.getElementById("page").innerText = "-";
            document.getElementById("sources").innerText = "-";
            document.getElementById("relevance").innerText = "-";


            try {

                const response = await fetch(
                    "/ask",
                    {
                        method: "POST",

                        headers: {
                            "Content-Type": "application/json"
                        },

                        body: JSON.stringify({
                            question: question
                        })
                    }
                );


                const data = await response.json();


                if (!response.ok) {

                    throw new Error(
                        data.detail ||
                        data.error ||
                        "Request failed"
                    );
                }


                document.getElementById(
                    "queryType"
                ).innerText =
                    data.query_type || "-";


                document.getElementById(
                    "scope"
                ).innerText =
                    data.scope || "-";


                document.getElementById(
                    "retrieval"
                ).innerText =
                    data.retrieval_strategy || "-";


                document.getElementById(
                    "page"
                ).innerText =
                    data.target_page ?? "-";


                document.getElementById(
                    "sources"
                ).innerText =
                    data.sources &&
                    data.sources.length
                        ? data.sources.join(", ")
                        : "-";


                document.getElementById(
                    "relevance"
                ).innerText =
                    data.relevance || "-";


                answer.innerText =
                    data.answer ||
                    "No answer returned.";


            } catch (error) {

                answer.innerText =
                    "Error: " + error.message;

            }


            answer.classList.remove("loading");
        }

        </script>

    </body>
    </html>
    """


@app.post("/ask")
def ask(request: QuestionRequest):

    question = request.question.strip()

    if not question:

        return {
            "error": "Question cannot be empty."
        }

    return adaptive_rag(question)


@app.get("/health")
def health():

    return {
        "status": "running",
        "system": "Multi-Mode Adaptive RAG"
    }


def find_free_port(start=8001):

    for port in range(start, start + 20):

        sock = socket.socket(
            socket.AF_INET,
            socket.SOCK_STREAM
        )

        try:

            sock.bind(
                ("127.0.0.1", port)
            )

            sock.close()

            return port

        except OSError:

            sock.close()

    raise RuntimeError(
        "No free port found."
    )


PORT = find_free_port()

print("Selected port:", PORT)


config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=PORT,
    log_level="warning"
)


server = uvicorn.Server(config)


def run_server():

    loop = asyncio.new_event_loop()

    asyncio.set_event_loop(loop)

    loop.run_until_complete(
        server.serve()
    )


thread = threading.Thread(
    target=run_server,
    daemon=True
)


thread.start()


time.sleep(3)


if thread.is_alive():

    print("FastAPI server started successfully.")
    print("Opening Adaptive RAG website...")

    try:

        output.serve_kernel_port_as_window(PORT)

    except Exception:

        output.serve_kernel_port_as_iframe(PORT)

else:

    raise RuntimeError(
        "FastAPI server failed to start."
    )

Selected port: 8001
FastAPI server started successfully.
Opening Adaptive RAG website...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>